### Imports

In [1]:
from sqlitesearch import TextSearchIndex
from rag_helper import RAGBase
from dotenv import load_dotenv
from google import genai
import os
from google.genai import types
import json

from openai import OpenAI

# Frameworks
from toyaikit.llm import OpenAIClient
from toyaikit.llm import OpenAIChatCompletionsClient
from toyaikit.tools import Tools
from toyaikit.chat import IPythonChatInterface
from toyaikit.chat.runners import OpenAIResponsesRunner, DisplayingRunnerCallback
from toyaikit.llm import OpenAIChatCompletionsClient
from toyaikit.chat.runners import OpenAIChatCompletionsRunner

In [2]:
GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")
client = genai.Client(api_key=GOOGLE_API_KEY)

In [3]:
# Point the OpenAI client at Google's Gemini OpenAI-compatible endpoint
gemini_client = OpenAI(
    api_key=os.getenv("GOOGLE_API_KEY"),
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/"
)


In [4]:
model="gemini-2.5-flash"

In [5]:
# Connect to the index database:
sqlite_index = TextSearchIndex(
    text_fields=["question", "section", "answer"],
    keyword_fields=["course"],
    db_path="faq.db"
)

In [6]:
# Check how many documents are in the index:
sqlite_index.count()

84

In [7]:
prompt = "Can I still join the course after it started?"
results = sqlite_index.search(prompt, num_results=5)
[doc["question"] for doc in results]

['I just discovered the course. Can I still join?',
 'I missed the first homework - can I still get a certificate?',
 'How do I start using Google Gemini models in the Module 1 notebook through the OpenAI-compatible endpoint?',
 'Homework: Why does the content keep changing?',
 'What happens to code saved in Codespaces if I do not commit it?']

### RAG with sqlitesearch



In [8]:
assistant = RAGBase(
    index=sqlite_index,
    llm_client=client
)

In [9]:
# test the llm
prompt = "How do I run Ollama locally?"
response = assistant.rag(prompt)
print(response)

To run Ollama locally, follow these steps:

1.  **Install Ollama**:
    *   Visit [https://ollama.com/download](https://ollama.com/download).
    *   Choose your operating system:
        *   **macOS**: Download and install the `.pkg` file.
        *   **Windows**: Download and install the `.msi` file.
        *   **Linux**: Run the following command in your terminal:
            ```bash
            curl -fsSL https://ollama.com/install.sh | sh
            ```

2.  **Start a model locally**:
    *   Once Ollama is installed, open a terminal.
    *   Run the following command to download the LLaMA 3 model and start it:
        ```bash
        ollama run llama3
        ```
        This command will download the ~4GB LLaMA 3 model, start it locally, and open a chat-like interface where you can type questions.

3.  **Test the Ollama local server (optional)**:
    *   To verify the server is running, open another terminal and execute:
        ```bash
        curl http://localhost:11434
    

In [10]:
# sqlite_index.close()

### Function-calling

In [11]:
#  let's see what the LLM does without any tools.
messages = [
    {"role": "user", "content": "I just discovered the course. Can I join it?"}
]
    
# Combine messages for the API call, preserving role information
combined_content = ""
for msg in messages:
    combined_content += f"[{msg['role'].upper()}]\n{msg['content']}\n\n"


response = client.models.generate_content(
    model=model,
    contents=combined_content,
)
print(response.text)

That's great you found a course you're interested in!

To give you the best possible answer, I need a little more information about the course. Could you please tell me:

1.  **What is the name of the course?**
2.  **Where is it offered?** (e.g., specific university, Coursera, edX, Udemy, a private platform, etc.)
3.  **Do you know its official start date or if it's currently running?**

In general, here's how you can usually check and what might be possible:

*   **If it hasn't started yet:** You'll likely just need to register or apply before the deadline.
*   **If it's a self-paced course (like many on Coursera, Udemy, etc.):** You can usually join at any time, even if it "started" weeks ago. You just work through the material at your own speed.
*   **If it's a cohort-based course (with live sessions, strict deadlines, or a specific group of students):**
    *   **If it just started:** You might be able to join and catch up. Check the course page for "late registration" options or c

The model answers from its general knowledge.
It doesn't know about our FAQ, so the answer is vague and not helpful. 
This is exactly why we need RAG, and why we want to hand the model a tool.

In [12]:
# Defining the tool
# First we define a top-level search function that queries the index directly. 
# The model will reference it by this name. 
def search(query: str) -> list:
    boost_dict = {"question": 3.0, "section": 0.5}
    filter_dict = {"course": "llm-zoomcamp"}

    return sqlite_index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
        filter_dict=filter_dict
    )

In [13]:
# Next we tell the model about this function. 
# The description is the most important field, because the model reads it to decide when to call the function. 
# parameters is a JSON schema for the arguments, and we mark query as required so the model always fills it in.
search_tool = {
    "type": "function",
    "name": "search", # name of the function
    "description": "Search the FAQ database for entries matching the given query.",
    "parameters": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "Search query text to look up in the course FAQ."
            }
        },
        "required": ["query"],
        "additionalProperties": False
    }
}

In [14]:
# using google search search_tool
google_search_tool = types.Tool(function_declarations=[
    types.FunctionDeclaration(
        name="search", #name of the function
        description="Search the FAQ database for entries matching the given query.",
        parameters=types.Schema(
            type="object",
            properties={
                "query": types.Schema(
                    type="string",
                    description="Search query text to look up in the course FAQ."
                )
            },
            required=["query"]
        )
    )
])

# Now we send the same question as before, but this time we include the tool in the request:
messages = [
    {"role": "user", "content": "I just discovered the course. Can I still join it?"}
]
    
# Combine messages for the API call, preserving role information
combined_content = ""
for msg in messages:
    combined_content += f"[{msg['role'].upper()}]\n{msg['content']}\n\n"

response = client.models.generate_content(
    model=model,
    contents=combined_content,
    config={"tools": [google_search_tool]}
)
response

GenerateContentResponse(
  candidates=[
    Candidate(
      content=Content(
        parts=[
          Part(
            function_call=FunctionCall(
              args=<... Max depth ...>,
              name=<... Max depth ...>
            ),
            thought_signature=b'\n\xe0\x01\x01\x11M2\x0f\xdc\xc08\xa3\xb7<\x92\xce+\x18q\xe6\xc4\x9c<\x14\x80\x89\xed)m\x90m\xebH\xe7\xf6o\xac\xe2Ay^M\x16\\\x7f8\xf6\x81\xc8\x1a\xb4\xf3\xa2\x95\xd1\xd9\xdd\xe3c\xd3\xf0\xca\xc8\xc8\xb4\xd5\xf5\xdf\x1c1\x9aMc.\xbc\xbf\x11\xda\x0ef\xbb 3\xdd\xefS\x9c\xaf\xbeq2u\x1c\xb7r\xcb\xa0...'
          ),
        ],
        role='model'
      ),
      finish_reason=<FinishReason.STOP: 'STOP'>,
      index=0
    ),
  ],
  model_version='gemini-2.5-flash',
  response_id='SX5haoqyLNOK7M8P8aju-Qs',
  sdk_http_response=HttpResponse(
    headers=<dict len=12>
  ),
  usage_metadata=GenerateContentResponseUsageMetadata(
    candidates_token_count=19,
    prompt_token_count=69,
    prompt_tokens_details=[
      Modalit

In [15]:
response.text

In [16]:
# Sending the question with the tool
# Now we send the same question as before, but this time we include the tool in the request:
messages = [
    {"role": "user", "content": "I just discovered the course. Can I still join it?"}
]
    
# Combine messages for the API call, preserving role information
combined_content = ""
for msg in messages:
    combined_content += f"[{msg['role'].upper()}]\n{msg['content']}\n\n"

response = client.models.generate_content(
    model=model,
    contents=combined_content,
    config={"tools": [search]}  # pass the function itself, not the dict
)

response

GenerateContentResponse(
  automatic_function_calling_history=[
    UserContent(
      parts=[
        Part(
          text="""[USER]
I just discovered the course. Can I still join it?

"""
        ),
      ],
      role='user'
    ),
    Content(
      parts=[
        Part(
          function_call=FunctionCall(
            args={<... 1 item at Max depth ...>},
            name='search'
          ),
          thought_signature=b'\n\xa2\x03\x01\x11M2\x0f\x07y\xab\xee\xcc\xdcT\xc2\xa6\x12i\xab\xc9\x89\x8a\x85]T\xef=0R\xc5\xe1\x00\x01\xb3R\x85\xff\xbc\xa4\xde\xc7\xed\x13\xc4\xecp\xe5\xe5_\x81\xd1}\xdc\x16\xc9\x00\x9f\xa5\xde\r\xe7\xf0t\x89"\x14\x84\xfbQ\xff\x84\xa8\x9ey\x8cUY\xaa\xc8\x06Ta[k0\xe7^\xe5\xfe\x9e\xed\t6\xe4\xd2b...'
        ),
      ],
      role='model'
    ),
    Content(
      parts=[
        Part(
          function_response=FunctionResponse(
            name='search',
            response={<... 1 item at Max depth ...>}
          )
        ),
      ],
      role='user'
 

In [17]:
response.text

'Yes, you can still join the course. However, if you want to receive a certificate, you need to submit your project while submissions are still being accepted. Certificates are only awarded if you finish the course with a "live" cohort because you need to peer-review other projects, which is only possible when the course is actively running.'

The function call contains JSON arguments. We parse them, call our search function, and serialize the result.

In [18]:
response.candidates[0].content

Content(
  parts=[
    Part(
      text='Yes, you can still join the course. However, if you want to receive a certificate, you need to submit your project while submissions are still being accepted. Certificates are only awarded if you finish the course with a "live" cohort because you need to peer-review other projects, which is only possible when the course is actively running.',
      thought_signature=b'\n\xc5\x05\x01\x11M2\x0fQ\\h\xda\xc0\xc6\x11\xcf\xfa-\x02\n\xa5"\x08>\xc4l<\xea\xd6\x94\x06^0\xfb\xb8\xa1ZS\n:^\xda{?Jl\x9d(+g\x15\nJ\xf9\x1a\xefY\xa6\x16a\x04\x8f\x84\x1b\xd2K\x1c\x19\x03\xcb\x06\xf7H\xfcGn\nQ\xe8z\xc9/\xb8\x1e*^\xa7\xa9\x08\xe7\xa4\tx\x19\xd4\xb2\xbf...'
    ),
  ],
  role='model'
)

In [19]:
call = response.automatic_function_calling_history[1].parts[0].function_call
call


FunctionCall(
  args={
    'query': 'Can I still join a course after discovering it?'
  },
  name='search'
)

In [20]:
args = dict(call.args)
args

{'query': 'Can I still join a course after discovering it?'}

In [21]:
results = search(**args)
results

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': '9f689c185f',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I missed the first homework - can I still get a certificate?',
  'answer': 'Yes, you need to pass the Capstone project to get the certificate. Homework is not mandatory, though it is recommended for reinforcing concepts, and the points awarded count towards your rank on the leaderboard.'},
 {'id': '69d122f12e',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
  'answer': 'No, you can only get a certificate if you finish the course with a "live" cohort.\n\n

In [22]:
result_json = json.dumps(results, indent=2)
result_json

'[\n  {\n    "id": "74eb249bbf",\n    "course": "llm-zoomcamp",\n    "section": "General Course-Related Questions",\n    "question": "I just discovered the course. Can I still join?",\n    "answer": "Yes, but if you want to receive a certificate, you need to submit your project while we\\u2019re still accepting submissions."\n  },\n  {\n    "id": "9f689c185f",\n    "course": "llm-zoomcamp",\n    "section": "General Course-Related Questions",\n    "question": "I missed the first homework - can I still get a certificate?",\n    "answer": "Yes, you need to pass the Capstone project to get the certificate. Homework is not mandatory, though it is recommended for reinforcing concepts, and the points awarded count towards your rank on the leaderboard."\n  },\n  {\n    "id": "69d122f12e",\n    "course": "llm-zoomcamp",\n    "section": "General Course-Related Questions",\n    "question": "Certificate: Can I follow the course in a self-paced mode and get a certificate?",\n    "answer": "No, you 

In [23]:
response.candidates[0].content

Content(
  parts=[
    Part(
      text='Yes, you can still join the course. However, if you want to receive a certificate, you need to submit your project while submissions are still being accepted. Certificates are only awarded if you finish the course with a "live" cohort because you need to peer-review other projects, which is only possible when the course is actively running.',
      thought_signature=b'\n\xc5\x05\x01\x11M2\x0fQ\\h\xda\xc0\xc6\x11\xcf\xfa-\x02\n\xa5"\x08>\xc4l<\xea\xd6\x94\x06^0\xfb\xb8\xa1ZS\n:^\xda{?Jl\x9d(+g\x15\nJ\xf9\x1a\xefY\xa6\x16a\x04\x8f\x84\x1b\xd2K\x1c\x19\x03\xcb\x06\xf7H\xfcGn\nQ\xe8z\xc9/\xb8\x1e*^\xa7\xa9\x08\xe7\xa4\tx\x19\xd4\xb2\xbf...'
    ),
  ],
  role='model'
)

### Asking the model again

In [24]:
# Now we send this result back to the model. 
# Rebuild messages entirely as types.Content objects so the SDK accepts them.
# Use automatic_function_calling_history[1] for the model's function call step.
messages = [
    types.Content(role="user", parts=[types.Part(text="I just discovered the course. Can I still join it?")]),
    response.automatic_function_calling_history[1],   # model's function_call Content
    types.Content(
        role="user",
        parts=[types.Part(
            function_response=types.FunctionResponse(
                name=call.name,
                response={"result": results}
            )
        )]
    )
]
messages

[Content(
   parts=[
     Part(
       text='I just discovered the course. Can I still join it?'
     ),
   ],
   role='user'
 ),
 Content(
   parts=[
     Part(
       function_call=FunctionCall(
         args={
           'query': 'Can I still join a course after discovering it?'
         },
         name='search'
       ),
       thought_signature=b'\n\xa2\x03\x01\x11M2\x0f\x07y\xab\xee\xcc\xdcT\xc2\xa6\x12i\xab\xc9\x89\x8a\x85]T\xef=0R\xc5\xe1\x00\x01\xb3R\x85\xff\xbc\xa4\xde\xc7\xed\x13\xc4\xecp\xe5\xe5_\x81\xd1}\xdc\x16\xc9\x00\x9f\xa5\xde\r\xe7\xf0t\x89"\x14\x84\xfbQ\xff\x84\xa8\x9ey\x8cUY\xaa\xc8\x06Ta[k0\xe7^\xe5\xfe\x9e\xed\t6\xe4\xd2b...'
     ),
   ],
   role='model'
 ),
 Content(
   parts=[
     Part(
       function_response=FunctionResponse(
         name='search',
         response={
           'result': [
             {<... 5 items at Max depth ...>},
             {<... 5 items at Max depth ...>},
             {<... 5 items at Max depth ...>},
             {<... 5 item

In [25]:
# Asking the model again with the full history including the search results
final_response = client.models.generate_content(
    model=model,
    contents=messages,
    config={"tools": [google_search_tool]}
)
print(final_response.text)

Yes, you can still join the course. However, if you want to receive a certificate, you need to submit your project while submissions are still being accepted.


### Agentic Loop

 #### developer prompt

In [26]:
# A developer prompt
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches.

Try to expand your search by using new keywords
based on the results you get from the search.

At the end, ask if there are other areas that the user wants to explore.
""".strip()

#### function-call helper

In [27]:
# We'll be running function calls repeatedly inside the loop, so let's wrap that in a small helper. 
# It turns the JSON arguments into a Python dict, calls the right function, and serializes the result. 
# We only have one tool for now, so we dispatch on the function name directly.
def make_call(call):
    args = dict(call.args)  # already a dict, no json.loads needed

    if call.name == "search":
        result = search(**args)

    return types.Content(
        role="user",
        parts=[types.Part(
            function_response=types.FunctionResponse(
                name=call.name,
                response={"result": result}
            )
        )]
    )


#### Processing one response

In [28]:
# Let's process a single model response. 
# We append each output entry to the conversation, print any messages, and run any function calls. 
# Function-call results get appended too.
# The has_function_calls flag tells us whether the model needs another API call. 
# If the response contains a function call, the updated messages has tool output the model hasn't seen yet. 
# We'll need to send it back.

question = "I just discovered the course. Can I join it?"

messages = [
    types.Content(role="user", parts=[types.Part(text=question)])
]

response = client.models.generate_content(
    model=model,
    contents=messages,
    config=types.GenerateContentConfig(
        system_instruction=instructions,   # "developer" role  = system_instruction in config
        tools=[google_search_tool]
    )
)

messages.append(response.candidates[0].content)   # extend → single append
has_function_calls = False

for part in response.candidates[0].content.parts:
    if part.function_call:                          # item.type == "function_call"
        print("function_call:", part.function_call.name, dict(part.function_call.args))
        call_output = make_call(part.function_call) # pass function_call, not the part
        messages.append(call_output)
        has_function_calls = True

    elif part.text:                                 # item.type == "message"
        print("ASSISTANT:")
        print(part.text)                            # item.content[0].text → part.text


function_call: search {'query': 'join course'}


#### The full agent loop

In [29]:
# We wrap this in a while loop. 
# The loop keeps calling the model until it returns a response without any function calls. 
# We also keep an iteration counter so we can see how many round-trips happened.

question = "I just discovered the course. Can I join it?"

messages = [
    types.Content(role="user", parts=[types.Part(text=question)])
]

it = 1

while True:
    print(f"iteration #{it}...")
    has_function_calls = False
    
    response = client.models.generate_content(
        model=model,
        contents=messages,
        config=types.GenerateContentConfig(
            system_instruction=instructions,   # "developer" role  = system_instruction in config
            tools=[google_search_tool]
        )
    )

    messages.append(response.candidates[0].content)   # extend → single append

    for part in response.candidates[0].content.parts:
        if part.function_call:                          # item.type == "function_call"
            print("function_call:", part.function_call.name, dict(part.function_call.args))
            call_output = make_call(part.function_call) # pass function_call, not the part
            messages.append(call_output)
            has_function_calls = True

        elif part.text:                                 # item.type == "message"
            print("ASSISTANT:")
            print(part.text)                            # item.content[0].text → part.text
   
    it = it + 1
    if has_function_calls == False:
        break

iteration #1...
function_call: search {'query': 'join course'}
function_call: search {'query': 'enroll course'}
iteration #2...
ASSISTANT:
Yes, you can still join the course. However, if you want to receive a certificate, you need to submit your project while submissions are still being accepted.

Is there anything else you would like to explore about the course?


In [30]:
# let's wrap it in a function
def agent_loop(instructions, question, model="gemini-2.5-flash") -> str:
    # We wrap this in a while loop. 
    # The loop keeps calling the model until it returns a response without any function calls. 
    # We also keep an iteration counter so we can see how many round-trips happened.

    messages = [
        types.Content(role="user", parts=[types.Part(text=question)])
    ]

    it = 1

    while True:
        print(f"iteration #{it}...")
        has_function_calls = False
        
        response = client.models.generate_content(
            model=model,
            contents=messages,
            config=types.GenerateContentConfig(
                system_instruction=instructions,   # "developer" role  = system_instruction in config
                tools=[google_search_tool]
            )
        )

        messages.append(response.candidates[0].content)   # extend → single append

        for part in response.candidates[0].content.parts:
            if part.function_call:                          # item.type == "function_call"
                print("function_call:", part.function_call.name, dict(part.function_call.args))
                call_output = make_call(part.function_call) # pass function_call, not the part
                messages.append(call_output)
                has_function_calls = True

            elif part.text:                                 # item.type == "message"
                print("ASSISTANT:")
                last_answer = part.text
                return last_answer
                print(last_answer)                            # item.content[0].text → part.text
    
        it = it + 1
        if has_function_calls == False:
            break
    

In [31]:
# Try it with a question that has a typo:
agent_loop(instructions, "How do I run Olama locally?")

iteration #1...
function_call: search {'query': 'How to run Olama locally'}
iteration #2...
ASSISTANT:


'To run Ollama locally, follow these steps:\n\n1.  **Install Ollama**:\n    *   Visit [https://ollama.com/download](https://ollama.com/download).\n    *   Choose your operating system (macOS, Windows, or Linux) and follow the installation instructions:\n        *   **macOS**: Download and install the `.pkg` file.\n        *   **Windows**: Download and install the `.msi` file.\n        *   **Linux**: Run the following command in your terminal:\n            '

In [32]:
agent_loop(instructions, "I just discovered the course. Can I still join it?")

iteration #1...
function_call: search {'query': 'join course'}
iteration #2...
ASSISTANT:


'Yes, you can still join the course. However, if you wish to receive a certificate, you need to submit your project while submissions are still being accepted.\n\nIs there anything else you would like to know?'

In [33]:
# Encouraging multiple searches
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches. First perform search, analyze the results 
and then perform more searches. 

At the end, ask if there are other areas that the user wants to explore.
""".strip()

agent_loop(instructions, "I just discovered the course. Can I join it?")

iteration #1...
function_call: search {'query': 'join course'}
iteration #2...
ASSISTANT:


'Yes, you can still join the course. However, if you wish to receive a certificate, you will need to submit your project while submissions are still being accepted.\n\nIs there anything else you would like to explore about the course?'

#### Restricting off-topic questions

In [34]:
agent_loop(instructions, "what's queen gambit?")

iteration #1...
function_call: search {'query': 'queen gambit'}
iteration #2...
function_call: search {'query': "Queen's Gambit"}
iteration #3...
ASSISTANT:


'I couldn\'t find a direct definition of "Queen\'s Gambit" in the course materials. It appears the term might have been mentioned in the context of discussions about APIs, token counting, or other technical topics within the `llm-zoomcamp` course, but not as a subject explained within the course itself.\n\nAre you looking for a general definition of "Queen\'s Gambit" (e.g., related to chess or the TV show), or were you expecting to find it discussed as a specific concept within the course?\n\nLet me know if there are other areas you\'d like to explore!'

In [35]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches. First perform search, analyze the results 
and then perform more searches. 

The question has to be about the course or its logistics, offtopic questions 
shouldn't be answered. If the search returns nothing, it's likely an off-topic question.
If you can't answer the question using FAQ, don't do it yourself. Only use the 
facts from the FAQ database.

At the end, ask if there are other areas that the user wants to explore.
""".strip()

agent_loop(instructions, "what's queen gambit?")

iteration #1...
function_call: search {'query': 'queen gambit'}
iteration #2...
ASSISTANT:


'I\'m sorry, but I can only answer questions related to the course or its logistics. "Queen gambit" does not appear to be related to the course material. Is there anything else I can help you with regarding the course?'

### Frameworks

In [36]:
# Registering the tool
# We register our search function along with the schema from earlier lessons:
agent_tools = Tools()
agent_tools.add_tool(search, google_search_tool)

In [37]:
# Letting ToyAIKit generate the schema
# If we add a type hint and a docstring to search, ToyAIKit reads them and derives the schema for us:
def search(query: str) -> dict[str, str]:
    boost_dict = {"question": 3.0, "section": 0.5}
    filter_dict = {"course": "llm-zoomcamp"}

    return sqlite_index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
        filter_dict=filter_dict
    )

In [38]:
# Then register it without passing a schema:
agent_tools = Tools()
agent_tools.add_tool(search)
agent_tools.get_tools()

[{'type': 'function',
  'name': 'search',
  'description': 'No description provided.',
  'parameters': {'type': 'object',
   'properties': {'query': {'type': 'string',
     'description': 'query parameter'}},
   'required': ['query'],
   'additionalProperties': False}}]

#### The chat interface and runner

In [39]:
llm_client = OpenAIChatCompletionsClient(
    model="gemini-2.5-flash",   # swap in whichever Gemini model you want
    client=gemini_client
)

In [40]:
# Create the chat interface and a callback, then build the runner:
# The chat_interface handles display in the notebook. 
# The callback renders model messages and tool calls as they happen.
# The runner runs the agent loop, the same while True we wrote by hand. 
# It sends messages, executes function calls, adds tool outputs back, and repeats until the model is done.
chat_interface = IPythonChatInterface()
callback = DisplayingRunnerCallback(chat_interface)

runner = OpenAIChatCompletionsRunner(
    tools=agent_tools,
    developer_prompt=instructions,
    chat_interface=chat_interface,
    llm_client=llm_client
)

In [41]:
# Running one prompt
result = runner.loop(
    prompt="How do I run Olama locally?",
    callback=callback,
)

-> Response received


-> Response received


In [42]:
result.cost

CostInfo(input_cost=Decimal('0.0005001'), output_cost=Decimal('0.0007925'), total_cost=Decimal('0.0012926'))

In [43]:
result.all_messages

[{'role': 'system',
  'content': "You're a course teaching assistant.\nYou're given a question from a course student and your task is to answer it.\n\nIf you want to look up information, use the search function. \nUse as many keywords from the user question as possible when making first requests.\n\nMake multiple searches. First perform search, analyze the results \nand then perform more searches. \n\nThe question has to be about the course or its logistics, offtopic questions \nshouldn't be answered. If the search returns nothing, it's likely an off-topic question.\nIf you can't answer the question using FAQ, don't do it yourself. Only use the \nfacts from the FAQ database.\n\nAt the end, ask if there are other areas that the user wants to explore."},
 {'role': 'user', 'content': 'How do I run Olama locally?'},
 ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='fun

#### Continuing the conversation

In [44]:
result2 = runner.loop(
    prompt="How do I run a different model?",
    previous_messages=result.all_messages,
    callback=callback,
)

-> Response received


-> Response received


The runner picks up where the last call left off, with the same agent loop and an extended history. 

#### Interactive chat

In [45]:
# runner.run()

### Other Frameworks

#### PydanticAI

- A type-safe agent framework that supports multiple LLM providers. 
- Tools are plain Python functions with type hints, no wrappers needed. 
- Switching providers is as simple as changing the model string.

#### LangChain / LangGraph

- A popular framework with lots of integrations. 
- LangChain handles the basics, and LangGraph adds graph-based workflows for more complex agent patterns.
- Good choice if you need lots of integrations (vector stores, document loaders, etc.) and a large community.

#### Google ADK

- The Agent Development Kit from Google. 
- If you plan to use Gemini models, this is the one I'd reach for. 
- It exposes the same building blocks we've seen, like tools, instructions, and sessions. 
- It also integrates with Google Cloud.
- Good choice if your stack is on Google Cloud or you specifically want Gemini.



#### Others
- CrewAI - multi-agent orchestration
- AutoGen - multi-agent conversations from Microsoft
- Semantic Kernel - from Microsoft, supports C# and Python
- Smolagents - lightweight agent framework from HuggingFace
- Anthropic Tool Use - Anthropic's native tool use API

In [46]:
# sqlite_index.close()